# 06 — CYP2D6 Outlier Check

**Roadmap Step 1**: does excluding CYP2D6 compounds
flagged by two independent, model-choice-agnostic criteria change CV performance for
the four CYP2D6-relevant configs from `05_cv_comparison`? All flagging and retraining
logic lives in `scripts/cyp2d6_outlier_check.py` (CLAUDE.md: scripts train, notebooks
report/analyze) -- **this notebook only loads and reports its already-completed
outputs; nothing here retrains or re-runs anything.**

**Two independent flagging criteria** (both computed from existing 05 artifacts --
no new training for the flagging step itself):

1. **CV-residual**: pooled out-of-fold |y_pred - y_true| for CYP2D6, from
   `chemprop_chemeleoninit`'s 25 (5 repeat x 5 fold) OOF prediction files only --
   deliberately not all 12 configs, so outlier status is model-independent once
   frozen. Threshold **confirmed by the user** after reviewing a candidate-threshold
   report: flag `mean_abs_residual` > the 95th percentile of the pooled distribution.
2. **CI-width**: flag the top 5% of CYP2D6 compounds by
   `(conf_high - conf_low)`, already in `train_inhibition_curated.csv`. As `04c`
   already found CI-width correlates with residual on every isoform, this criterion
   can only show whether removing noisy CYP2D6 points helps CYP2D6 -- not why CYP2D6
   specifically underperforms.

Threshold confirmation happened **before** any retraining (non-negotiable per the
task spec, to keep this a real test rather than data-dredging) -- flagged-compound
counts and overlap were reported and confirmed first; only then did retraining run.

**Exclusion design** (also confirmed with the user before retraining): exclusion is
**train-only** -- a flagged compound is dropped from whichever fold's training pool
it would have been part of, but still scored normally whenever it lands in a fold's
held-out test set. This keeps the "before" and "after" evaluation population
identical (the same 1,493 CYP2D6-labeled compounds, the same 25 folds), so any metric
change reflects the retrained model, not a smaller/easier test set. For
`chemprop_chemeleoninit`'s multitask model, exclusion masks only the CYP2D6 label for
flagged compounds -- they keep contributing to CYP1A2/2C9/3A4 training.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from scipy.stats import shapiro, ttest_rel, wilcoxon

OUT = REPO_ROOT / "outputs" / "06_outlier_check"
FLAGGED_DIR = OUT / "flagged_compounds"
SCORE_DIR_06 = OUT / "scores"
SUMMARY_05_PATH = REPO_ROOT / "outputs" / "05_cv_comparison" / "summary_table.csv"

CONFIGS = ["chemeleon__rf", "chemeleon__lightgbm", "ecfp4_narrow__lightgbm", "chemprop_chemeleoninit"]
CRITERIA = ["residual", "ci_width"]
METRICS = ["ST-RAE", "R2"]
ALPHA = 0.05
N_CYP2D6_LABELED = 1493  # confirmed against train_inhibition_curated.csv in the flagging script's own log

print(f"python: {sys.version.split()[0]}")
print(f"configs: {CONFIGS}")
print(f"criteria: {CRITERIA}")

python: 3.11.13
configs: ['chemeleon__rf', 'chemeleon__lightgbm', 'ecfp4_narrow__lightgbm', 'chemprop_chemeleoninit']
criteria: ['residual', 'ci_width']


## 1. Flagging results

Both flagged-compound lists were written by `scripts/cyp2d6_outlier_check.py` to
`outputs/06_outlier_check/flagged_compounds/` before any retraining. Loaded here, not
recomputed.

In [2]:
residual_flagged = pd.read_csv(FLAGGED_DIR / "criterion1_residual_flagged.csv")
ci_flagged = pd.read_csv(FLAGGED_DIR / "criterion2_ci_width_flagged.csv")

overlap = set(residual_flagged["inchikey"]) & set(ci_flagged["inchikey"])
smaller = min(len(residual_flagged), len(ci_flagged))

flag_summary = pd.DataFrame([
    {"criterion": "1. CV-residual (mean_abs_residual > p95, chemprop_chemeleoninit OOF)",
     "n_flagged": len(residual_flagged), "pct_of_1493": len(residual_flagged) / N_CYP2D6_LABELED},
    {"criterion": "2. CI-width (top 5% of conf_high - conf_low)",
     "n_flagged": len(ci_flagged), "pct_of_1493": len(ci_flagged) / N_CYP2D6_LABELED},
])
print(flag_summary.to_string(index=False))
print(f"\noverlap: {len(overlap)} compounds ({len(overlap) / smaller:.2%} of the smaller 75-compound set)")

                                                           criterion  n_flagged  pct_of_1493
1. CV-residual (mean_abs_residual > p95, chemprop_chemeleoninit OOF)         75     0.050234
                        2. CI-width (top 5% of conf_high - conf_low)         75     0.050234

overlap: 26 compounds (34.67% of the smaller 75-compound set)


**What this shows:** criterion 1 (CV-residual, p95) and criterion 2 (CI-width, top
5%) each flag 75 of the 1,493 CYP2D6-labeled compounds (5.02% -- p95 of a
1,493-compound distribution and a fixed 5% quantile land on the same count here by
coincidence of the sample size, not by construction). 26 compounds (34.7% of the
smaller 75-compound set) are flagged by both -- a real but partial overlap, consistent
with 04c's finding that CI-width correlates with residual without being the same
signal: most of what each criterion flags, the other does not.

## 2. Before/after CV comparison

**Statistical method -- a confirmed deviation from 05, not silently adapted.** 05's
Levene's -> RM-ANOVA/Tukey-or-Friedman/Conover-Friedman pipeline (with BH correction
above 10 groups and a 5-metric Bonferroni gate) was built for a many-config, 5-metric
cross-sectional screen, where Levene's homogeneity-of-variance check makes sense
across *independent* groups. This comparison is different in kind: for each
(config, criterion), "baseline" and "excluded" are the **same 25 folds**, retrained
with vs. without the flagged compounds -- a paired, not independent, design. Levene's
test is the wrong assumption check for that (flagged by the user before this section
was written, in place of the author's first proposal to reuse Levene's-based
branching literally). Method actually used, per (config, criterion, metric):

1. Compute the 25 per-fold deltas (excluded - baseline).
2. Shapiro-Wilk test on those 25 deltas.
3. Paired t-test (on the raw before/after arrays) if Shapiro-Wilk does not reject
   normality (p >= 0.05); Wilcoxon signed-rank on the deltas otherwise.

No BH correction and no cross-metric Bonferroni gate (both calibrated for 05's joint
5-metric, many-group screen, not this 2-metric before/after check) -- confirmed with
the user. Only ST-RAE and R2 are tested (the two metrics the task asks about); each
(config, criterion) pair gets its own test at alpha=0.05, uncorrected across the 4
configs x 2 criteria -- with 16 total tests below, ~0.8 false positives are expected
under the null by chance alone, worth keeping in mind when reading the table.

"Before" point estimates -- the mean of each fold's 1,000-sample bootstrap
distribution, matching 05's own aggregation convention exactly -- come from 05's own
`summary_table.csv`, unchanged. "After" point estimates are aggregated the identical
way from `scripts/cyp2d6_outlier_check.py`'s own per-fold score files.

In [3]:
rows = []
for criterion in CRITERIA:
    for config in CONFIGS:
        for repeat in range(5):
            for fold in range(5):
                path = SCORE_DIR_06 / f"{criterion}__{config}__repeat{repeat}_fold{fold}.csv"
                df = pd.read_csv(path)
                # mean of the 1,000-sample bootstrap distribution -- matches 05's own aggregation convention
                pe = df.groupby("Endpoint")[METRICS].mean()
                row = {"criterion": criterion, "config": config, "repeat": repeat, "fold": fold}
                for m in METRICS:
                    row[f"after_{m}"] = pe.loc["CYP2D6_pIC50_direct_inhibition", m]
                rows.append(row)
after_df = pd.DataFrame(rows)
print(f"after_df: {after_df.shape} (expect (200, 6): 4 configs x 2 criteria x 25 folds)")

summary_05 = pd.read_csv(SUMMARY_05_PATH)

results = []
for criterion in CRITERIA:
    for config in CONFIGS:
        sub_after = (after_df[(after_df["criterion"] == criterion) & (after_df["config"] == config)]
                     .sort_values(["repeat", "fold"]).reset_index(drop=True))
        sub_before = summary_05[summary_05["config"] == config].sort_values(["repeat", "fold"]).reset_index(drop=True)
        assert len(sub_after) == 25 and len(sub_before) == 25, "expected 25 folds per (config, criterion)"
        assert (sub_after[["repeat", "fold"]].to_numpy() == sub_before[["repeat", "fold"]].to_numpy()).all(), \
            "before/after fold order mismatch -- paired test would be invalid"

        for metric in METRICS:
            before = sub_before[f"CYP2D6_{metric}"].to_numpy()
            after = sub_after[f"after_{metric}"].to_numpy()
            delta = after - before

            shapiro_p = float(shapiro(delta).pvalue)
            normal = shapiro_p >= ALPHA
            if normal:
                test_name = "paired t-test"
                _, p_value = ttest_rel(after, before)
            else:
                test_name = "Wilcoxon signed-rank"
                _, p_value = wilcoxon(delta)

            results.append({
                "criterion": criterion, "config": config, "metric": metric,
                "before_mean": before.mean(), "after_mean": after.mean(), "delta_mean": delta.mean(),
                "shapiro_p": shapiro_p, "branch": "parametric" if normal else "nonparametric",
                "test": test_name, "p_value": float(p_value), "significant": bool(p_value < ALPHA),
            })

results_df = pd.DataFrame(results)
results_path = OUT / "before_after_comparison.csv"
results_df.to_csv(results_path, index=False)
print(f"wrote {results_path}")

pd.set_option("display.width", 160)
results_df

after_df: (200, 6) (expect (200, 6): 4 configs x 2 criteria x 25 folds)
wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/06_outlier_check/before_after_comparison.csv


,criterion,config,metric,before_mean,after_mean,delta_mean,shapiro_p,branch,test,p_value,significant
0,residual,chemeleon__rf,ST-RAE,0.937235,0.902073,-0.035161,0.780878,parametric,paired t-test,4.597603e-08,True
1,residual,chemeleon__rf,R2,0.141597,0.135805,-0.005792,0.634752,parametric,paired t-test,1.532456e-01,False
2,residual,chemeleon__lightgbm,ST-RAE,0.948368,0.930195,-0.018174,0.475966,parametric,paired t-test,5.133234e-04,True
3,residual,chemeleon__lightgbm,R2,0.110424,0.106032,-0.004392,0.800465,parametric,paired t-test,4.522349e-01,False
4,residual,ecfp4_narrow__lightgbm,ST-RAE,0.959821,0.946349,-0.013472,0.543563,parametric,paired t-test,5.175929e-02,False
5,residual,ecfp4_narrow__lightgbm,R2,0.099610,0.089870,-0.009740,0.514681,parametric,paired t-test,8.807214e-02,False
6,residual,chemprop_chemeleoninit,ST-RAE,1.003907,0.934307,-0.069600,0.268430,parametric,paired t-test,1.561109e-05,True
7,residual,chemprop_chemeleoninit,R2,0.097991,0.112979,0.014988,0.178114,parametric,paired t-test,2.075844e-01,False
8,ci_width,chemeleon__rf,ST-RAE,0.937235,0.928985,-0.008250,0.334973,parametric,paired t-test,8.834666e-02,False
9,ci_width,chemeleon__rf,R2,0.141597,0.120066,-0.021531,0.423237,parametric,paired t-test,2.086402e-05,True


**What this shows:**

**Criterion 1 (CV-residual, model-independent outliers):** ST-RAE improves after
exclusion in all 4 configs, significantly so in 3 of 4 (`chemeleon__rf` p=4.6e-08,
`chemeleon__lightgbm` p=5.1e-04, `chemprop_chemeleoninit` p=1.6e-05); `ecfp4_narrow__
lightgbm` trends the same direction but falls just short of alpha=0.05 (p=0.052). R2
moves in small, non-significant, mixed directions in every config. **Removing the
compounds the model itself struggled with, pooled across repeats, improves CYP2D6
ST-RAE without a detectable R2 cost, fairly consistently across model families.**

**Criterion 2 (CI-width, measurement-uncertainty outliers):** the picture is
different. ST-RAE reaches significance in only one config -- `chemprop_chemeleoninit`
(p=1.5e-05, a similarly-sized improvement to its own criterion-1 result); the three
tabular configs show small, non-significant, mixed-direction ST-RAE changes. R2,
meanwhile, gets **significantly worse** for all three tabular configs
(`chemeleon__rf` p=2.1e-05, `chemeleon__lightgbm` p=4.2e-06, `ecfp4_narrow__lightgbm`
p=0.017) and is flat for `chemprop_chemeleoninit` (p=0.974). **Removing the
widest-CI compounds does not reliably help CYP2D6 ST-RAE for the tabular configs, and
measurably hurts their R2** -- consistent with the task's own framing that, given
04c's residual/CI-width correlation, this criterion tests "does removing noisy points
help" rather than "why does CYP2D6 underperform," and here the answer for CI-width
specifically is: for three of four configs, no on ST-RAE and actively worse on R2.

`chemprop_chemeleoninit` is the one config that benefits (significantly, on ST-RAE)
from *either* criterion -- suggesting it is sensitive to CYP2D6's hard/noisy
compounds regardless of which of the two criteria identifies them, unlike the tabular
configs, whose R2 the CI-width criterion specifically damages.

Both are genuine, reportable results, not failed runs: criterion 1 is a real,
model-independent-outlier finding that a specific subset of CYP2D6 compounds hurts
ST-RAE across model families; criterion 2 is a real null-to-negative result --
CYP2D6's characteristically wide-CI compounds are not simply prunable noise from the
tabular models' point of view.

## 3. Outcome

- **Criterion 1 (CV-residual) exclusion helps CYP2D6 ST-RAE**: significant
  improvement for `chemeleon__rf`, `chemeleon__lightgbm`, and
  `chemprop_chemeleoninit`; a same-direction, non-significant trend for
  `ecfp4_narrow__lightgbm`. R2 is not detectably affected either way, in any config.
- **Criterion 2 (CI-width) exclusion does not reliably help CYP2D6 ST-RAE**: only
  `chemprop_chemeleoninit` shows a significant ST-RAE improvement; the three tabular
  configs do not. R2 **significantly worsens** for all three tabular configs and is
  flat for `chemprop_chemeleoninit`.
- The two criteria diverge in practical outcome despite a genuine 34.7% overlap in
  which compounds they flag -- most of each criterion's flagged set is unique to it,
  and that non-overlapping majority is where their effects differ.